# 🏃 Maratona di Berlino: il caldo ti rallenta davvero?

**Intuizione:** "Se fa più caldo, i maratoneti vanno più piano."

Oggi lo verifichiamo con i dati reali della Maratona di Berlino (1974–2019).

**Cosa faremo:**
1. Scaricare due file da Kaggle
2. Unirli in un unico dataset
3. Esplorarlo con pandas
4. Creare grafici per rispondere alla domanda
5. Tirare le conclusioni

## 1. Carica i file

Da [Kaggle](https://www.kaggle.com/datasets/aiaiaidavid/berlin-marathons-data) abbiamo scaricato due CSV:
- **Berlin_Marathon_data_1974_2019.csv** — tutti i corridori di ogni edizione (~885.000 righe!)
- **Berlin_Marathon_weather_data_since_1974.csv** — il meteo del giorno della gara (46 righe)

Nel mondo reale i dati arrivano quasi sempre così: **separati**. Tocca a noi unirli.

In [ ]:
# --- LIBRERIE ---
# pandas: per lavorare con tabelle di dati (come Excel, ma in Python)
# matplotlib: per creare grafici
# numpy: per calcoli matematici (es. linea di tendenza)
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Carica i file dal tuo computer
# files.upload() apre una finestra per selezionare i file dal PC
from google.colab import files
uploaded = files.upload()  # seleziona ENTRAMBI i CSV

In [ ]:
# pd.read_csv() legge un file CSV e lo trasforma in un DataFrame
# Un DataFrame è una tabella con righe e colonne, come un foglio Excel
# low_memory=False: il file è grande, diciamo a pandas di caricarlo tutto in memoria
runners = pd.read_csv('Berlin_Marathon_data_1974_2019.csv', low_memory=False)
weather = pd.read_csv('Berlin_Marathon_weather_data_since_1974.csv')

# .shape restituisce (righe, colonne) del DataFrame
print(f'Corridori: {runners.shape[0]} righe, {runners.shape[1]} colonne')
print(f'Meteo: {weather.shape[0]} righe, {weather.shape[1]} colonne')

## 2. Diamo un'occhiata ai dati

Prima regola della data science: **guarda sempre i dati prima di fare qualsiasi cosa**.

In [ ]:
# .head() mostra le prime 5 righe del DataFrame
# È il primo comando da usare SEMPRE quando apri un dataset
# Serve a capire: che colonne ci sono? Che tipo di dati contengono?
runners.head()

In [ ]:
# Stessa cosa per il meteo
# Nota: NaN significa "Not a Number" = dato mancante
# È normale! Nel mondo reale i dati hanno sempre dei buchi
weather.head()

## 3. Dal caos all'ordine: costruiamo il nostro dataset

Abbiamo 885.000 corridori ma a noi serve solo il **vincitore di ogni anno**.

Poi dobbiamo **unire** quel risultato con il meteo.

### Passo 1: troviamo il vincitore di ogni anno

Il vincitore è chi ha il tempo più basso. Con `groupby` raggruppiamo per anno e prendiamo il minimo.

In [ ]:
# groupby('YEAR'): raggruppa tutte le righe che hanno lo stesso anno
# ['TIME'].min(): per ogni gruppo, prendi il tempo più basso (= il vincitore)
# .reset_index(): trasforma il risultato in un DataFrame normale
#
# Pensatelo così: è come fare una tabella pivot in Excel
# "Per ogni anno, dimmi il tempo minimo"
winners = runners.groupby('YEAR')['TIME'].min().reset_index()

# Rinominiamo le colonne per chiarezza
winners.columns = ['YEAR', 'WINNER_TIME']

print(f'Vincitori trovati: {len(winners)} edizioni')
winners.head(10)

### Passo 2: uniamo vincitori e meteo (merge)

I due dataset hanno una colonna in comune: **YEAR**.

`merge` funziona come un puzzle: prende ogni anno e attacca le informazioni meteo accanto al vincitore.

```
Vincitori:                    Meteo:
YEAR | WINNER_TIME            YEAR | AVG_TEMP_C | ...
1974 | 02:44:53               1974 | 5.4        | ...
1975 | 02:47:08               1975 | 14.3       | ...
        \                        /
         merge(on='YEAR')
              |
YEAR | WINNER_TIME | AVG_TEMP_C | ...
1974 | 02:44:53    | 5.4        | ...
1975 | 02:47:08    | 14.3       | ...
```

In [ ]:
# .merge() unisce due DataFrame su una colonna in comune
# on='YEAR': la colonna che fa da "ponte" tra le due tabelle
# Per ogni anno, pandas cerca la riga corrispondente nell'altra tabella
# e le attacca insieme
df = winners.merge(weather, on='YEAR')

print(f'Risultato: {len(df)} righe, {len(df.columns)} colonne')
df.head(10)

### Passo 3: convertiamo il tempo in minuti

Il tempo è scritto come testo (es. `"02:44:53"`). Per fare calcoli e grafici ci serve un **numero**.

Esempio: `"02:44:53"` → 2×60 + 44 + 53/60 = **164.88 minuti**

In [ ]:
# Creiamo una funzione che converte "HH:MM:SS" in minuti
# def = definisce una funzione (un blocco di codice riutilizzabile)
# La funzione prende una stringa come "02:44:53" e restituisce un numero
def tempo_in_minuti(t):
    parti = str(t).split(':')   # split(':') divide la stringa sui ":"
                                 # "02:44:53" diventa ['02', '44', '53']
    ore = int(parti[0])          # int() converte testo in numero intero
    minuti = int(parti[1])
    secondi = int(parti[2])
    return ore * 60 + minuti + secondi / 60

# .apply() esegue la funzione su OGNI riga della colonna
# È come trascinare una formula in Excel su tutta la colonna
df['WINNER_MINUTES'] = df['WINNER_TIME'].apply(tempo_in_minuti)

# Controlliamo che funzioni: il tempo in testo e in minuti devono corrispondere
df[['YEAR', 'WINNER_TIME', 'WINNER_MINUTES', 'AVG_TEMP_C']].head()

### Riepilogo: cosa abbiamo fatto

```
885.000 corridori          46 righe meteo
       \                      /
    groupby(YEAR)         così com'è
        \                    /
         \                  /
          merge(on=YEAR)
               |
        44 righe pulite
    vincitore + meteo + minuti
```

In [ ]:
# .describe() calcola le statistiche di base per ogni colonna numerica:
# count = quanti valori ci sono (se < 44, ci sono dati mancanti)
# mean = media       std = deviazione standard (quanto i valori variano)
# min/max = minimo e massimo
# 25%/50%/75% = quartili (il 50% è la mediana)
df.describe()

## 4. La domanda chiave: il caldo rallenta i maratoneti?

Per rispondere usiamo uno **scatter plot** (grafico a dispersione):
- Ogni **punto** è un'edizione della maratona
- Asse X = temperatura media del giorno
- Asse Y = tempo del vincitore in minuti

Se l'intuizione è giusta, dovremmo vedere i punti **salire verso destra** (più caldo → tempo più alto → più lento).

In [ ]:
# plt.figure() crea un nuovo grafico
# figsize=(10, 6): larghezza 10 pollici, altezza 6 (cambia le dimensioni)
plt.figure(figsize=(10, 6))

# plt.scatter() crea uno scatter plot
# Primo argomento: valori asse X (temperatura)
# Secondo argomento: valori asse Y (tempo vincitore)
# color: il colore dei pallini (provate a cambiarlo! es. 'red', 'green', '#FF5733')
# s=80: dimensione dei pallini (provate s=20 o s=200 per vedere la differenza)
# alpha=0.7: trasparenza (1=opaco, 0=invisibile) — utile quando i punti si sovrappongono
plt.scatter(df['AVG_TEMP_C'], df['WINNER_MINUTES'], color='steelblue', s=80, alpha=0.7)

# --- ETICHETTE E FORMATTAZIONE ---
# Senza queste righe il grafico funziona, ma nessuno capisce cosa sta guardando!
# xlabel/ylabel: etichette degli assi (cosa rappresentano X e Y)
# fontsize: dimensione del testo in punti
plt.xlabel('Temperatura media (°C)', fontsize=13)
plt.ylabel('Tempo vincitore (minuti)', fontsize=13)

# title: il titolo del grafico — deve rispondere alla domanda "cosa sto guardando?"
plt.title('Maratona di Berlino: temperatura vs tempo del vincitore', fontsize=14)

# grid: mostra una griglia di sfondo per leggere meglio i valori
# alpha=0.3: la griglia è quasi trasparente, non distrae dai dati
plt.grid(True, alpha=0.3)

# tight_layout: ricalcola i margini per evitare che xlabel/ylabel vengano tagliate
# Senza questa riga, a volte le scritte escono fuori dal grafico
plt.tight_layout()

# show: "stampa" il grafico a schermo
# In Colab non è strettamente necessario, ma è buona pratica metterlo sempre
plt.show()

### Aggiungiamo una linea di tendenza

A occhio è difficile capire la direzione. Aggiungiamo una **retta di regressione** che "segue" i punti.

Se la retta sale → più caldo = più lento (intuizione confermata).  
Se la retta è piatta o scende → la temperatura non c'entra (intuizione smentita).

In [ ]:
# np.polyfit() calcola la retta che meglio si adatta ai punti
# Il "1" finale significa: polinomio di grado 1 = una retta (y = ax + b)
# coeff contiene [a, b] dove a=pendenza, b=intercetta
coeff = np.polyfit(df['AVG_TEMP_C'], df['WINNER_MINUTES'], 1)

# np.poly1d() crea una funzione dalla retta, così possiamo calcolare i punti
linea = np.poly1d(coeff)

# np.linspace() crea 100 punti equidistanti tra il minimo e il massimo della temperatura
# Servono per disegnare la retta liscia
x_linea = np.linspace(df['AVG_TEMP_C'].min(), df['AVG_TEMP_C'].max(), 100)

plt.figure(figsize=(10, 6))

# Disegniamo DUE cose sullo stesso grafico:
# 1) I punti reali (scatter)
plt.scatter(df['AVG_TEMP_C'], df['WINNER_MINUTES'], color='steelblue', s=80, alpha=0.7, label='Dati reali')

# 2) La retta di tendenza (plot = linea)
# linewidth=2: spessore della linea
# linestyle='--': tratteggiata (altre opzioni: '-' continua, ':' puntini, '-.' tratto-punto)
# label: il testo che apparirà nella legenda
plt.plot(x_linea, linea(x_linea), color='red', linewidth=2, linestyle='--', label='Tendenza')

plt.xlabel('Temperatura media (°C)', fontsize=13)
plt.ylabel('Tempo vincitore (minuti)', fontsize=13)
plt.title('Più caldo = più lento? Vediamo la tendenza', fontsize=14)

# legend() mostra la legenda: un riquadro che spiega cosa rappresenta ogni elemento
# Funziona solo se avete messo label='...' nei plot sopra
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- INTERPRETIAMO LA RETTA ---
# coeff[0] è la pendenza (slope):
#   positivo → la retta sale (più caldo = più lento)
#   negativo → la retta scende (più caldo = più veloce)
#   vicino a 0 → la retta è piatta (la temperatura non influenza)
print(f'Pendenza della retta: {coeff[0]:.2f}')
if coeff[0] > 0:
    print(f'Ogni grado in più, il vincitore ci mette {coeff[0]:.2f} minuti in più')
else:
    print(f'Ogni grado in più, il vincitore ci mette {abs(coeff[0]):.2f} minuti in MENO')
    print('Sorpresa! La retta scende: la temperatura non sembra rallentare i vincitori')

### Quanto è forte questa relazione?

La **correlazione** ci dice quanto due variabili vanno "a braccetto":
- **+1** = perfettamente insieme (una sale, l'altra pure)
- **-1** = perfettamente opposti (una sale, l'altra scende)
- **0** = nessuna relazione

Nella vita reale raramente si vedono valori sopra 0.7 o sotto -0.7.

In [ ]:
# --- CORRELAZIONE ---
# La correlazione di Pearson misura quanto due variabili si muovono insieme
# È un singolo numero che riassume la relazione tra due colonne
#
# Come funziona:
#   .corr() prende tutti i valori di AVG_TEMP_C e WINNER_MINUTES,
#   e calcola quanto "vanno nella stessa direzione"
#
# Cosa restituisce:
#   +1.0 = correlazione perfetta positiva (una sale, l'altra SEMPRE sale)
#    0.0 = nessuna correlazione (le due variabili sono indipendenti)
#   -1.0 = correlazione perfetta negativa (una sale, l'altra SEMPRE scende)
#
# Nella realtà i valori perfetti (+1 o -1) non esistono mai.
# Come regola pratica:
#   |corr| < 0.3  → debole (c'è poco legame)
#   |corr| 0.3-0.7 → moderata (c'è un legame ma non fortissimo)
#   |corr| > 0.7  → forte (le due variabili sono molto legate)

corr = df['AVG_TEMP_C'].corr(df['WINNER_MINUTES'])
print(f'Correlazione temperatura-tempo: {corr:.3f}')
# :.3f = formatta il numero con 3 decimali (f = float)
print()

# --- INTERPRETIAMO IL RISULTATO ---

# 1) Il segno ci dice la DIREZIONE
if corr > 0:
    print('DIREZIONE: Positiva')
    print('  Quando la temperatura sale, il tempo tende a salire (= più lento)')
else:
    print('DIREZIONE: Negativa')
    print('  Quando la temperatura sale, il tempo tende a scendere (= più veloce)')
print()

# 2) Il valore assoluto ci dice la FORZA
# abs() toglie il segno: abs(-0.5) = 0.5, abs(0.3) = 0.3
forza = abs(corr)
if forza < 0.3:
    print(f'FORZA: DEBOLE ({forza:.3f})')
    print('  La relazione esiste appena. Altri fattori contano molto di più.')
    print('  Esempio: è come dire "i ragazzi con scarpe rosse vanno meglio a scuola"')
    print('  Forse è vero sui dati, ma è un caso, non una vera causa.')
elif forza < 0.7:
    print(f'FORZA: MODERATA ({forza:.3f})')
    print('  C\'è una relazione reale, ma non è determinante.')
    print('  La temperatura influenza, ma tanti altri fattori contano.')
else:
    print(f'FORZA: FORTE ({forza:.3f})')
    print('  Le due variabili sono molto legate.')

## 5. I tempi migliorano negli anni?

Un'altra intuizione: con allenamenti migliori e tecnologia, i maratoneti dovrebbero essere più veloci oggi che negli anni '70.

Questa volta usiamo un **line plot** (grafico a linea): perfetto per mostrare come un valore cambia nel tempo.

In [ ]:
# plt.plot() crea un grafico a linea (connette i punti in ordine)
# marker='o': mostra un pallino su ogni punto dati
plt.figure(figsize=(10, 6))
plt.plot(df['YEAR'], df['WINNER_MINUTES'], marker='o', color='green', linewidth=1.5, markersize=6)
plt.xlabel('Anno', fontsize=13)
plt.ylabel('Tempo vincitore (minuti)', fontsize=13)
plt.title('Evoluzione dei tempi: i runner migliorano?', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# .iloc[0] = prima riga, .iloc[-1] = ultima riga
primo = df.iloc[0]
ultimo = df.iloc[-1]
diff = primo['WINNER_MINUTES'] - ultimo['WINNER_MINUTES']
print(f"Nel {int(primo['YEAR'])} il vincitore ha fatto {primo['WINNER_MINUTES']:.1f} minuti")
print(f"Nel {int(ultimo['YEAR'])} il vincitore ha fatto {ultimo['WINNER_MINUTES']:.1f} minuti")
print(f'Miglioramento: {diff:.1f} minuti in meno!')

## 6. Il grafico che racconta tutto

Mettiamo **tre variabili** in un unico grafico:
- Asse X = temperatura
- Asse Y = tempo del vincitore
- Colore = anno

Questo ci aiuta a capire se il pattern temperatura→tempo è reale o se è mascherato dal miglioramento negli anni.

In [ ]:
# --- SCATTER PLOT A 3 VARIABILI ---
# Finora abbiamo messo 2 variabili su un grafico (X e Y).
# Ma possiamo aggiungere una TERZA variabile usando il COLORE.
#
# È come avere 3 dimensioni su un foglio 2D:
#   Asse X = temperatura
#   Asse Y = tempo del vincitore
#   Colore = anno (viola = vecchio, giallo = recente)

plt.figure(figsize=(10, 6))

# La differenza rispetto a plt.scatter() normale:
#   - NON usiamo color='steelblue' (colore fisso)
#   - Usiamo c=df['YEAR'] (il colore dipende dal valore dell'anno)
#   - cmap='viridis' è la "mappa colori": una scala che va dal viola al giallo
#     (altre opzioni: 'coolwarm' blu→rosso, 'plasma', 'inferno')
scatter = plt.scatter(
    df['AVG_TEMP_C'],       # asse X
    df['WINNER_MINUTES'],   # asse Y
    c=df['YEAR'],           # colore = anno
    cmap='viridis',         # scala di colori
    s=80,
    alpha=0.8
)

# colorbar: aggiunge la barra laterale che spiega cosa significano i colori
# Senza colorbar, nessuno saprebbe che viola=1974 e giallo=2019
plt.colorbar(scatter, label='Anno')

plt.xlabel('Temperatura media (°C)', fontsize=13)
plt.ylabel('Tempo vincitore (minuti)', fontsize=13)
plt.title('Temperatura vs Tempo, colorato per anno', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# COSA CERCARE NEL GRAFICO:
# - I punti viola (anni '70-'80) sono in ALTO → tempi peggiori
# - I punti gialli (anni recenti) sono in BASSO → tempi migliori
# - Ma non c'è un pattern chiaro sinistra→destra (freddo→caldo)
# Questo conferma: il fattore dominante è l'ANNO, non la temperatura

## 7. Conclusioni

Cosa abbiamo scoperto?

1. **L'intuizione NON è confermata** — la relazione temperatura→tempo è debolissima (correlazione vicina a 0). Il caldo non sembra rallentare i vincitori in modo significativo.
2. **Il vero fattore è il tempo storico** — i runner moderni (punti gialli) sono molto più veloci dei runner degli anni '70 (punti viola), indipendentemente dalla temperatura.
3. **L'intuizione non è inutile** — ci ha dato una domanda da verificare. Ma i dati ci dicono che la realtà è più complessa di quello che pensavamo.

**Lezione chiave:** 
- A volte i dati **smentiscono** la nostra intuizione, e va bene così — è il bello della data science
- Quando una variabile (l'anno) ha un effetto enorme, può **nascondere** l'effetto di un'altra (la temperatura)
- Questo si chiama **variabile confondente** (confounding variable)

---

## Sfide

Prova a rispondere con il codice:

1. **Facile:** Qual è stato l'anno più caldo? E il più freddo? Che tempo ha fatto il vincitore in quei due anni?
2. **Media:** Crea un bar chart con la temperatura media per decade (anni '70, '80, '90, 2000, 2010)
3. **Difficile:** Dividi i dati in due gruppi (sotto e sopra 15°C) e confronta il tempo medio dei vincitori

In [ ]:
# SFIDA 1: Anno più caldo e più freddo
# Suggerimento: usa idxmax() e idxmin()
# df['AVG_TEMP_C'].idxmax() restituisce l'INDICE della riga con il valore massimo
# df.loc[indice] restituisce la riga intera


In [ ]:
# SFIDA 2: Temperatura media per decade
# Suggerimento: crea una colonna 'DECADE' con df['YEAR'] // 10 * 10
# L'operatore // è la divisione intera: 1987 // 10 = 198, * 10 = 1980
# Poi usa groupby('DECADE')['AVG_TEMP_C'].mean()


In [ ]:
# SFIDA 3: Confronto sotto/sopra 15°C
# Suggerimento: filtra con df[df['AVG_TEMP_C'] < 15]
# df[condizione] restituisce solo le righe dove la condizione è True
# Poi calcola .mean() su WINNER_MINUTES per entrambi i gruppi
